# IMDb Movie Recommender — Encodage & Choix de features (Notebook)

Objectif : **valider et optimiser l'encodage** (genres / acteurs / réalisateurs / scénaristes / genre_tokens) **avant** de figer `train.py`.

Contraintes :
- Pas d'affichage massif : `head(10)` max, échantillons limités.
- On prend des décisions **data-driven** (mesures + petits tests NN).


## 1) Setup

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

# --- Robust project root detection ---
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "config.py").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

print("PROJECT_ROOT =", PROJECT_ROOT)

sys.path.insert(0, str(PROJECT_ROOT))
import config as cfg

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

PROJECT_ROOT = /home/gau/projets/movie-recommender


## 2) Charger le dataset (parquet)

In [2]:
data_path = PROJECT_ROOT / cfg.DATA_PATH
print("Using:", data_path)

df = pd.read_parquet(data_path).reset_index(drop=True)
print("shape:", df.shape)

df.head(10)

Using: /home/gau/projets/movie-recommender/data/processed/movie_imdb.parquet
shape: (99068, 14)


,tconst,primaryTitle,originalTitle,startYear,runtimeMinutes,genres,genre_tokens,actors,directors,writers,people,averageRating,numVotes,content
0,tt0000009,Miss Jerry,Miss Jerry,1894,45,Romance,g:Romance,Blanche_Bayliss William_Courtenay Chauncey_Depew,Alexander_Black,Alexander_Black,Alexander_Black Blanche_Bayliss William_Courtenay Chauncey_Depew Alexander_B...,5.2,232,Miss Jerry Romance Alexander_Black Blanche_Bayliss William_Courtenay Chaunce...
1,tt0000574,The Story of the Kelly Gang,The Story of the Kelly Gang,1906,70,Action Adventure Biography,g:Action g:Adventure g:Biography gpair:Action|Adventure gpair:Action|Biograp...,Elizabeth_Tait John_Tait Nicholas_Brierley,Charles_Tait,Charles_Tait,Charles_Tait Elizabeth_Tait John_Tait Nicholas_Brierley Charles_Tait,6.0,1046,The Story of the Kelly Gang Action Adventure Biography Charles_Tait Elizabet...
2,tt0001892,Den sorte drøm,Den sorte drøm,1911,53,Drama,g:Drama,Asta_Nielsen Valdemar_Psilander Gunnar_Helsengreen,Urban_Gad,Urban_Gad Gebhard_Schätzler-Perasini,Urban_Gad Asta_Nielsen Valdemar_Psilander Gunnar_Helsengreen Urban_Gad Gebha...,5.9,292,Den sorte drøm Drama Urban_Gad Asta_Nielsen Valdemar_Psilander Gunnar_Helsen...
3,tt0002101,Cleopatra,Cleopatra,1912,100,Drama History,g:Drama g:History gpair:Drama|History,Helen_Gardner Pearl_Sindelar Miss_Fielding,Charles_L._Gaskill,Victorien_Sardou,Charles_L._Gaskill Helen_Gardner Pearl_Sindelar Miss_Fielding Victorien_Sardou,5.1,663,Cleopatra Drama History Charles_L._Gaskill Helen_Gardner Pearl_Sindelar Miss...
4,tt0002130,Dante's Inferno,L'inferno,1911,71,Adventure Drama Fantasy,g:Adventure g:Drama g:Fantasy gpair:Adventure|Drama gpair:Adventure|Fantasy ...,Salvatore_Papa Arturo_Pirovano Giuseppe_de_Liguoro,Francesco_Bertolini,Dante_Alighieri,Francesco_Bertolini Salvatore_Papa Arturo_Pirovano Giuseppe_de_Liguoro Dante...,7.1,4027,Dante's Inferno Adventure Drama Fantasy Francesco_Bertolini Salvatore_Papa A...
5,tt0002199,From the Manger to the Cross,From the Manger to the Cross,1912,71,Biography Drama,g:Biography g:Drama gpair:Biography|Drama,R._Henderson_Bland Percy_Dyer Gene_Gauntier,Sidney_Olcott,Gene_Gauntier,Sidney_Olcott R._Henderson_Bland Percy_Dyer Gene_Gauntier Gene_Gauntier,5.9,700,From the Manger to the Cross Biography Drama Sidney_Olcott R._Henderson_Blan...
6,tt0002423,Passion,Madame DuBarry,1919,113,Biography Drama Romance,g:Biography g:Drama g:Romance gpair:Biography|Drama gpair:Biography|Romance ...,Pola_Negri Emil_Jannings Harry_Liedtke,Ernst_Lubitsch,Norbert_Falk Hanns_Kräly,Ernst_Lubitsch Pola_Negri Emil_Jannings Harry_Liedtke Norbert_Falk Hanns_Kräly,6.7,1113,Passion Biography Drama Romance Ernst_Lubitsch Pola_Negri Emil_Jannings Harr...
7,tt0002445,Quo Vadis?,Quo Vadis?,1913,120,Drama History,g:Drama g:History gpair:Drama|History,Amleto_Novelli Gustavo_Serena Carlo_Cattaneo,Enrico_Guazzoni,Henryk_Sienkiewicz Enrico_Guazzoni,Enrico_Guazzoni Amleto_Novelli Gustavo_Serena Carlo_Cattaneo Henryk_Sienkiew...,6.1,519,Quo Vadis? Drama History Enrico_Guazzoni Amleto_Novelli Gustavo_Serena Carlo...
8,tt0002452,The Independence of Romania,Independenta Romaniei,1912,120,History War,g:History g:War gpair:History|War,Aristide_Demetriade Constanta_Demetriade Constantin_Nottara,Aristide_Demetriade,Aristide_Demetriade Petre_Liciu,Aristide_Demetriade Aristide_Demetriade Constanta_Demetriade Constantin_Nott...,6.3,284,The Independence of Romania History War Aristide_Demetriade Aristide_Demetri...
9,tt0002646,Atlantis,Atlantis,1913,121,Drama,g:Drama,Olaf_Fønss Ida_Orloff Ebba_Thomsen,August_Blom,Axel_Garde Gerhart_Hauptmann,August_Blom Olaf_Fønss Ida_Orloff Ebba_Thomsen Axel_Garde Gerhart_Hauptmann,6.5,522,Atlantis Drama August_Blom Olaf_Fønss Ida_Orloff Ebba_Thomsen Axel_Garde Ger...


In [79]:
# Échantillon aléatoire (petit) pour inspection rapide
df.sample(6, random_state=42)[["primaryTitle","startYear","averageRating","numVotes","genres","directors","writers","actors","genre_tokens"]].head(10)

,primaryTitle,startYear,averageRating,numVotes,genres,directors,writers,actors,genre_tokens
57973,Succubus,2024,4.7,2004,Drama Horror Thriller,R.J._Daniel_Hanna,R.J._Daniel_Hanna,Brendan_Bradley Olivia_Grace_Applegate Rosanna_Arquette,g:Drama g:Horror g:Thriller gpair:Drama|Horror gpair:Drama|Thriller gpair:Ho...
40250,Khakee,2004,7.4,17380,Action Crime Drama,Rajkumar_Santoshi,Ranjit_Kapoor Amit_Pachori,Amitabh_Bachchan Akshay_Kumar Ajay_Devgn,g:Action g:Crime g:Drama gpair:Action|Crime gpair:Action|Drama gpair:Crime|D...
50843,The Penthouse,2021,3.1,479,Crime Mystery Thriller,Massimiliano_Cerchi,David_Schifter,David_Schifter Vanessa_Ore Michael_Paré,g:Crime g:Mystery g:Thriller gpair:Crime|Mystery gpair:Crime|Thriller gpair:...
44232,A Time to Love,2005,6.4,401,Drama Romance,Jianqi_Huo,Wu_Si Renjie_Zhang,Wei_Zhao Yi_Lu Minjie_Cui,g:Drama g:Romance gpair:Drama|Romance
91086,Friends by Chance,2017,7.0,1606,Comedy Drama,Francesco_Bruni,Francesco_Bruni,Andrea_Carpenzano Giuliano_Montaldo Arturo_Bruni,g:Comedy g:Drama gpair:Comedy|Drama
34123,House of Terrors,1965,6.2,316,Horror Mystery,Hajime_Satô,Franco_Dal_Cer Hajime_Takaiwa,Kô_Nishimura Kô_Nishimura Masumi_Harukawa,g:Horror g:Mystery gpair:Horror|Mystery


## 3) Sanity checks rapides

In [3]:
needed_cols = ["primaryTitle","startYear","averageRating","numVotes","genres","genre_tokens","directors","writers","actors"]
missing = [c for c in needed_cols if c not in df.columns]
print("Missing columns:", missing)

df[["averageRating","numVotes","runtimeMinutes"]].describe(percentiles=[.5,.75,.9,.95,.99])

Missing columns: []


,averageRating,numVotes,runtimeMinutes
count,99068.000000,99068.0,99068.0
mean,5.897254,12383.072536,101.415674
std,1.252634,68730.754656,22.044601
min,1.000000,200.0,26.0
50%,6.100000,853.0,97.0
75%,6.800000,2947.0,110.0
90%,7.300000,13989.3,130.0
95%,7.600000,43622.25,144.0
99%,8.300000,253376.8,170.0
max,9.900000,3150021.0,360.0


## 4) Choisir `min_votes` intelligemment

In [65]:
import numpy as np
import pandas as pd
from datetime import datetime

# ---------------------------
# 4.1) Features: âge + votes/an + version bayésienne
# ---------------------------
def add_votes_time_features(df: pd.DataFrame, current_year: int | None = None, tau_years: float = 3.0):
    """
    Ajoute :
      - age_years
      - votes_per_year
      - votes_per_year_bayes  (lissé bayésien, anti-biais films récents)
    tau_years = "années fictives" (prior). 3.0 est un bon défaut.
    """
    out = df.copy()

    if current_year is None:
        current_year = datetime.now().year

    year = pd.to_numeric(out["startYear"], errors="coerce")
    votes = pd.to_numeric(out["numVotes"], errors="coerce").fillna(0).astype(float)

    # âge >= 1 (on inclut l'année de sortie) pour éviter division par 0
    age = (current_year - year).clip(lower=0)
    age = (age + 1).fillna(1).astype(float)

    out["age_years"] = age
    out["votes_per_year"] = votes / out["age_years"]

    # prior global sur votes_per_year
    mu_vpy = float(out["votes_per_year"].median()) if len(out) else 0.0

    # Bayes smoothing:
    # votes_per_year_bayes = (mu*tau + votes) / (age + tau)
    out["votes_per_year_bayes"] = (mu_vpy * tau_years + votes) / (out["age_years"] + tau_years)

    # log versions (robustes)
    out["log_votes"] = np.log1p(votes)
    out["log_vpy"] = np.log1p(out["votes_per_year"])
    out["log_vpy_bayes"] = np.log1p(out["votes_per_year_bayes"])

    return out, mu_vpy


# ---------------------------
# 4.2) Suggestion "intelligente" de seuil
# ---------------------------
def suggest_threshold(
    series: pd.Series,
    thresholds=None,
    w_coverage: float = 0.50,
    w_size: float = 0.20,
    w_quality: float = 0.30,
):
    """
    Choisit un seuil t sur `series` en optimisant :
      - coverage (somme conservée / somme totale)
      - size_ratio (#items conservés / total)
      - quality (moyenne conservée, normalisée)
    """
    x = pd.to_numeric(series, errors="coerce").fillna(0).to_numpy(dtype=float)
    n_total = len(x)
    total_signal = float(x.sum())
    xmax = float(x.max()) if float(x.max()) > 0 else 1.0

    if thresholds is None:
        xpos = x[x > 0]
        lo = float(np.percentile(xpos, 5)) if len(xpos) else 0.1
        hi = float(np.percentile(xpos, 99.5)) if len(xpos) else 10.0
        hi = max(hi, lo * 1.01)

        # seuils log-spaced + quelques valeurs humaines
        thresholds = np.unique(np.round(np.logspace(np.log10(max(lo, 1e-6)), np.log10(hi), 40), 4)).tolist()
        thresholds = sorted(set(thresholds + [0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 100]))

    rows = []
    for t in thresholds:
        mask = x >= t
        n = int(mask.sum())
        size_ratio = n / n_total if n_total else 0.0

        kept_signal = float(x[mask].sum()) if n else 0.0
        coverage = kept_signal / total_signal if total_signal > 0 else 0.0

        mean_kept = float(x[mask].mean()) if n else 0.0
        mean_norm = float(np.log1p(mean_kept) / np.log1p(xmax))

        score = w_coverage * coverage + w_size * size_ratio + w_quality * mean_norm

        rows.append({
            "threshold": float(t),
            "n_movies": n,
            "size_ratio": size_ratio,
            "coverage": coverage,
            "mean_kept": mean_kept,
            "score": score,
        })

    tbl = pd.DataFrame(rows).sort_values("threshold").reset_index(drop=True)

    # --- ignore plateau where nothing is filtered ---
    tbl_eff = tbl[tbl["size_ratio"] < 0.999].copy()
    if not tbl_eff.empty:
        tbl = tbl_eff
    best = float(tbl.loc[tbl["score"].idxmax(), "threshold"])
    return best, tbl



# ---------------------------
# 4.3) RUN
# ---------------------------
df_v, mu_vpy = add_votes_time_features(df, tau_years=3.0)
print("mu votes_per_year =", round(mu_vpy, 3))

# A) seuil brut sur numVotes (en log pour stabilité)
best_votes_log, tbl_votes_log = suggest_threshold(df_v["log_votes"])
print("best threshold on log_votes =", best_votes_log)

# B) seuil sur votes/an (bayésien) -> recommandé pour ne pas pénaliser les films récents
best_vpy_bayes, tbl_vpy_bayes = suggest_threshold(df_v["votes_per_year_bayes"])
print("best threshold on votes_per_year_bayes =", best_vpy_bayes)

mu votes_per_year = 53.748
best threshold on log_votes = 5.4205
best threshold on votes_per_year_bayes = 5.0


In [66]:
tbl_vpy_bayes["no_filter"] = (tbl_vpy_bayes["size_ratio"] >= 0.9999)
tbl_vpy_bayes.sort_values("score", ascending=False).head(15)[
    ["threshold","n_movies","size_ratio","no_filter","coverage","mean_kept","score"]
]

,threshold,n_movies,size_ratio,no_filter,coverage,mean_kept,score
5,5.0000,97632,0.985505,False,0.999908,703.452233,0.861719
6,7.1178,94113,0.949984,False,0.999596,729.527675,0.855372
7,8.7688,91180,0.920378,False,0.999257,752.738964,0.850067
8,10.0000,89182,0.900210,False,0.998984,769.392849,0.846445
9,10.8028,87938,0.887653,False,0.998796,780.129813,0.844187
10,13.3086,84337,0.851304,False,0.998165,812.925817,0.837635
11,16.3957,80456,0.812129,False,0.997328,851.424714,0.830542
12,20.0000,76111,0.768270,False,0.996178,898.993055,0.822559
13,20.1988,75879,0.765928,False,0.996110,901.680288,0.822131
14,24.8841,70724,0.713893,False,0.994420,965.761934,0.812602


In [67]:
min_votes = 200
mask = pd.to_numeric(df["numVotes"], errors="coerce").fillna(0) >= min_votes
print("n_movies kept:", int(mask.sum()), "| ratio:", round(mask.mean(), 3))

n_movies kept: 99068 | ratio: 1.0


In [68]:
print("Chosen min_votes (final):", best_vpy_bayes)

Chosen min_votes (final): 5.0


## Section 4 — Dataset filtering (min_votes & temporal bias)

### Objective
Define a **robust filtering strategy** that:
- removes extremely obscure movies (noise),
- preserves diversity,
- does **not penalize recent movies** that did not yet have time to accumulate votes.

---

### Base filter: minimum number of votes

We fix a base threshold:

- **min_votes = 200**

Rationale:
- standard IMDb-like threshold,
- removes extreme noise,
- keeps ~99k movies (large diversity),
- stable baseline for the recommender.

---

### Temporal bias issue

Raw `numVotes` unfairly penalizes:
- recent releases,
- niche movies with strong early engagement.

To correct this, we introduce **votes per year**, smoothed with a Bayesian prior.

---

### Bayesian votes-per-year smoothing

We define:

- `age_years = current_year - startYear + 1`
- `votes_per_year = numVotes / age_years`

To avoid inflation for very recent movies, we apply Bayesian smoothing:

\[
votes\_per\_year\_bayes =
\frac{\mu_{vpy} \cdot \tau + numVotes}{age\_years + \tau}
\]

Where:
- \(\mu_{vpy}\) is the **median** votes-per-year over the dataset,
- \(\tau = 3\) years is a temporal prior.

Using the median ensures robustness to blockbusters and outliers.

---

### Final filtering rule

A movie is kept if **at least one** condition is satisfied:

- `numVotes >= 200`
- `votes_per_year_bayes >= 5`

Rationale:
- protects recent movies with real traction,
- preserves long-tail diversity,
- removes films that are both old **and** invisible.

---

### Final parameters (Section 4)

```text
MIN_VOTES = 200
VOTES_PER_YEAR_BAYES_TAU = 3.0
MIN_VOTES_PER_YEAR_BAYES = 5.0


## 5) Choisir `RATING_PRIOR` et `CONFIDENCE_K` (data-driven)

In [69]:
def suggest_prior_k(df: pd.DataFrame, q_prior: float = 0.50, q_k: float = 0.50, floor: int = 50):
    votes = pd.to_numeric(df["numVotes"], errors="coerce").fillna(0).to_numpy(dtype=float)
    votes = votes[votes > 0]
    if len(votes) == 0:
        return 150, 150
    prior = int(max(floor, np.quantile(votes, q_prior)))
    k = int(max(floor, np.quantile(votes, q_k)))
    return prior, k

prior_med, k_med = suggest_prior_k(df, 0.50, 0.50)
prior_strict, k_strict = suggest_prior_k(df, 0.60, 0.75)
prior_med, k_med, prior_strict, k_strict

(853, 853, 1295, 2947)

In [70]:
# Visual check rapide (pas indispensable)
global_mean = float(pd.to_numeric(df["averageRating"], errors="coerce").fillna(0).mean())
votes = pd.to_numeric(df["numVotes"], errors="coerce").fillna(0).astype(float)
ratings = pd.to_numeric(df["averageRating"], errors="coerce").fillna(0).astype(float)

def add_bayes_conf(df_in: pd.DataFrame, prior: float, k: float) -> pd.DataFrame:
    out = df_in.copy()
    out["rating_bayes"] = (global_mean * prior + votes * ratings) / (prior + votes)
    out["confidence"] = votes / (votes + k)
    return out

df_bc = add_bayes_conf(df, prior=prior_med, k=k_med)
df_bc[["averageRating","rating_bayes","confidence","numVotes"]].describe().round(3)

,averageRating,rating_bayes,confidence,numVotes
count,99068.000,99068.000,99068.000,99068.0
mean,5.897,5.981,0.545,12383.073
std,1.253,0.729,0.259,68730.755
min,1.000,1.055,0.190,200.0
25%,5.200,5.600,0.308,379.0
50%,6.100,5.978,0.500,853.0
75%,6.800,6.352,0.776,2947.0
max,9.900,9.299,1.000,3150021.0


### Decision (Section 5)

We fix:
- RATING_PRIOR = 500
- CONFIDENCE_K = 500

Rationale:
- Values are close to dataset medians (~850 votes) but slightly lower to preserve diversity.
- rating_bayes smooths extremes without collapsing variance.
- confidence correctly reflects reliability from 200 votes upward.


## 6) Diagnostic tokens (diversité / entropie)

In [71]:
from collections import Counter

def token_stats(series: pd.Series, name: str, top_k: int = 15):
    tokens = []
    for x in series.fillna("").astype(str):
        tokens.extend(x.split())
    c = Counter(tokens)
    total = sum(c.values()) if c else 1
    probs = np.array([v / total for v in c.values()], dtype=float)
    entropy = float(-(probs * np.log2(probs + 1e-12)).sum())
    print(f"\n{name}")
    print("-" * 50)
    print(f"Unique tokens: {len(c):,}")
    print(f"Entropy: {entropy:.2f}")
    print(f"Top {top_k} tokens:")
    for t, v in c.most_common(top_k):
        print(f"  {t}: {v}")

for col in ["directors","writers","actors","genre_tokens"]:
    if col in df.columns:
        token_stats(df[col], col)


directors
--------------------------------------------------
Unique tokens: 38,613
Entropy: 14.45
Top 15 tokens:
  Jesús_Franco: 98
  Michael_Curtiz: 90
  Richard_Thorpe: 74
  Lloyd_Bacon: 72
  John_Ford: 71
  William_Beaudine: 69
  Lesley_Selander: 68
  Gordon_Douglas: 66
  Raoul_Walsh: 62
  Cheh_Chang: 62
  Mervyn_LeRoy: 61
  Priyadarshan: 60
  Takashi_Miike: 58
  Norman_Taurog: 57
  William_A._Wellman: 56

writers
--------------------------------------------------
Unique tokens: 72,461
Entropy: 15.47
Top 15 tokens:
  Jing_Wong: 101
  Kuang_Ni: 95
  William_Shakespeare: 92
  Jesús_Franco: 77
  Robin_Bhatt: 62
  Jean-Claude_Carrière: 61
  Leonardo_Benvenuti: 59
  Erdogan_Tünas: 59
  Woody_Allen: 55
  Ben_Hecht: 53
  Stephen_King: 53
  Luc_Besson: 53
  Robert_E._Kent: 52
  Michel_Audiard: 52
  Paruchuri_Venkateswara_Rao: 52

actors
--------------------------------------------------
Unique tokens: 109,292
Entropy: 15.66
Top 15 tokens:
  Mohanlal: 198
  Mammootty: 170
  Amitabh_Bachchan

## 7) Tests NN par bloc (feature seule)

In [72]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

def test_block_nn(
    df: pd.DataFrame,
    col: str,
    query_title: str,
    min_df: int = 2,
    max_features: int = 5000,
    n_neighbors: int = 10,
):
    # Find query row (pick most voted if multiple)
    q = query_title.strip().lower()
    sub = df[df["primaryTitle"].fillna("").astype(str).str.lower() == q]
    if sub.empty:
        sub = df[df["primaryTitle"].fillna("").astype(str).str.lower().str.contains(q, regex=False)]
    if sub.empty:
        print(f"Query not found: {query_title}")
        return
    sub = sub.copy()
    sub["numVotes"] = pd.to_numeric(sub["numVotes"], errors="coerce").fillna(0)
    row = sub.sort_values("numVotes", ascending=False).iloc[0]
    idx = int(row.name)

    vec = TfidfVectorizer(
        lowercase=False,
        min_df=min_df,
        max_features=max_features,
        token_pattern=r"(?u)\b\w+\b",
    )
    X = vec.fit_transform(df[col].fillna("").astype(str))
    nnz = X[idx].nnz
    n_tokens = len(str(df.loc[idx, col]).split())
    print(f"  debug: n_tokens={n_tokens} | nnz={nnz} | vocab={len(vec.vocabulary_)}")
    nn = NearestNeighbors(metric="cosine", n_neighbors=n_neighbors)
    nn.fit(X)

    dist, ind = nn.kneighbors(X[idx])
    print(f"\nBlock: {col} | query: {df.loc[idx,'primaryTitle']} ({df.loc[idx,'startYear']})")
    for d, i in zip(dist[0][1:], ind[0][1:]):
        print(f"  - {df.iloc[i]['primaryTitle']} ({df.iloc[i]['startYear']}) | sim={1-float(d):.3f}")

# Run a few key blocks for a sanity pass
for args in [
    ("directors","Inception",2,5000),
    ("writers","Inception",2,5000),
    ("genre_tokens","Inception",1,5000),
]:
    col, title, mindf, mf = args
    if col in df.columns:
        test_block_nn(df, col, title, min_df=mindf, max_features=mf)

  debug: n_tokens=1 | nnz=1 | vocab=5000

Block: directors | query: Inception (2010)
  - Dunkirk (2017) | sim=1.000
  - Oppenheimer (2023) | sim=1.000
  - Inception (2010) | sim=1.000
  - The Dark Knight Rises (2012) | sim=1.000
  - Interstellar (2014) | sim=1.000
  - The Prestige (2006) | sim=1.000
  - The Dark Knight (2008) | sim=1.000
  - Batman Begins (2005) | sim=1.000
  - Insomnia (2002) | sim=1.000
  debug: n_tokens=1 | nnz=1 | vocab=5000

Block: writers | query: Inception (2010)
  - Dunkirk (2017) | sim=1.000
  - Oppenheimer (2023) | sim=1.000
  - Inception (2010) | sim=1.000
  - The Dark Knight Rises (2012) | sim=1.000
  - Interstellar (2014) | sim=1.000
  - The Prestige (2006) | sim=1.000
  - The Dark Knight (2008) | sim=1.000
  - Following (1998) | sim=1.000
  - Memento (2000) | sim=1.000
  debug: n_tokens=6 | nnz=6 | vocab=26

Block: genre_tokens | query: Inception (2010)
  - Dakota Bound (2001) | sim=1.000
  - Star Trek: Horizon (2016) | sim=1.000
  - Jurassic Park (1993) 

## 8) Prototype encodage complet (comme `train.py`)

In [73]:
from scipy import sparse
from sklearn.preprocessing import MultiLabelBinarizer

def split_genres(series: pd.Series) -> list[list[str]]:
    return (
        series.fillna("")
        .astype(str)
        .str.replace(",", " ", regex=False)
        .str.strip()
        .str.split()
        .tolist()
    )

# Suggested TF-IDF params (adjust based on block tests)
params_directors = dict(min_df=2, max_features=5000)
params_writers   = dict(min_df=2, max_features=8000)
params_actors    = dict(min_df=10, max_features=15000)
params_gtokens   = dict(min_df=1, max_features=5000)

# Suggested weights (tune as needed)
W = dict(genres=1.0, directors=2.0, writers=1.3, actors=0.6, genre_tokens=0.8)

# Encoders
mlb = MultiLabelBinarizer(sparse_output=True)
X_genres = mlb.fit_transform(split_genres(df["genres"]))

vec_dir = TfidfVectorizer(lowercase=False, token_pattern=r"(?u)\b\w+\b", **params_directors)
vec_wri = TfidfVectorizer(lowercase=False, token_pattern=r"(?u)\b\w+\b", **params_writers)
vec_act = TfidfVectorizer(lowercase=False, token_pattern=r"(?u)\b\w+\b", **params_actors)
vec_gt  = TfidfVectorizer(lowercase=False, token_pattern=r"(?u)\b\w+\b", **params_gtokens)

X_dir = vec_dir.fit_transform(df["directors"].fillna("").astype(str))
X_wri = vec_wri.fit_transform(df["writers"].fillna("").astype(str))
X_act = vec_act.fit_transform(df["actors"].fillna("").astype(str))
X_gt  = vec_gt.fit_transform(df["genre_tokens"].fillna("").astype(str))

X = sparse.hstack([
    W["genres"]       * X_genres,
    W["directors"]    * X_dir,
    W["writers"]      * X_wri,
    W["actors"]       * X_act,
    W["genre_tokens"] * X_gt,
], format="csr")

print("Combined X shape:", X.shape)

Combined X shape: (99068, 18606)


In [74]:
# Quick NN test on combined encoding
from sklearn.neighbors import NearestNeighbors

def recommend_from_X(df: pd.DataFrame, X, title: str, k: int = 10):
    q = title.strip().lower()
    sub = df[df["primaryTitle"].fillna("").astype(str).str.lower() == q]
    if sub.empty:
        sub = df[df["primaryTitle"].fillna("").astype(str).str.lower().str.contains(q, regex=False)]
    if sub.empty:
        print(f"❌ Query not found: {title}")
        return
    sub = sub.copy()
    sub["numVotes"] = pd.to_numeric(sub["numVotes"], errors="coerce").fillna(0)
    row = sub.sort_values("numVotes", ascending=False).iloc[0]
    idx = int(row.name)

    nn = NearestNeighbors(metric="cosine", n_neighbors=k+1)
    nn.fit(X)
    dist, ind = nn.kneighbors(X[idx])

    print(f"\nCombined encoding | query: {df.loc[idx,'primaryTitle']} ({df.loc[idx,'startYear']})")
    r = 0
    for d, i in zip(dist[0], ind[0]):
        if int(i) == idx:
            continue
        r += 1
        print(f"{r:2d}. {df.iloc[int(i)]['primaryTitle']} ({df.iloc[int(i)]['startYear']}) | sim={1-float(d):.3f}")
        if r >= k:
            break

recommend_from_X(df, X, "Inception", k=10)


Combined encoding | query: Inception (2010)
 1. Tenet (2020) | sim=0.848
 2. Interstellar (2014) | sim=0.798
 3. Dunkirk (2017) | sim=0.707
 4. The Prestige (2006) | sim=0.680
 5. The Dark Knight (2008) | sim=0.655
 6. The Dark Knight Rises (2012) | sim=0.655
 7. Roar: Tigers of the Sundarbans (2014) | sim=0.613
 8. My iz budushchego 2 (2010) | sim=0.613
 9. 2101 (2014) | sim=0.613
10. Mercury Man (2006) | sim=0.613
